In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

In [ ]:
dataset_list = ["ARC_Challenge", 
                "CommonSenseQA", "MMLU", "OpenBookQA"
                ]
model_list = [
    "EleutherAI/pythia-410",
    "EleutherAI/pythia-1b",
    "EleutherAI/pythia-1.4b",
    "meta-llama/Llama-3.2-1B",
    "meta-llama/Llama-3.2-3B",
    "Qwen/Qwen1.5-0.5B",
    "Qwen/Qwen1.5-1.8B",
    "Qwen/Qwen1.5-4B",
    "openai-community/gpt2",
    "openai-community/gpt2-large",
    "openai-community/gpt2-medium",
  ]

In [87]:
def get_mean_std(data): 
    mean = np.mean(data, axis=0)
    std = np.std(data, axis=0)
    return mean, std

In [88]:
def get_data_list(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        results = [json.loads(line) for line in f]

    all_grad_list = np.stack([r["grads_norm_list"] for r in results], axis=0)              # shape: (num_samples, dim)
    all_delta_z_list = np.stack([r["delta_z_norm_list"] for r in results], axis=0)
 
    all_grad_list = np.array(all_grad_list)
    all_delta_z_list = np.array(all_delta_z_list)
    
    upper_bound_list = all_grad_list * all_delta_z_list

    grad_mean_list, grad_std_list = get_mean_std(all_grad_list)
    delta_z_mean_list, delta_z_std_list = get_mean_std(all_delta_z_list)
    upper_bound_mean_list, upper_bound_std_list = get_mean_std(upper_bound_list)

    delta_log_prob_norm_list = [r["delta_log_prob_norm"] for r in results]

    result_dict = {
        "grad_mean_list": grad_mean_list[1:],
        "grad_std_list": grad_std_list[1:],
        "delta_z_mean_list": delta_z_mean_list[1:],
        "delta_z_std_list": delta_z_std_list[1:],
        "upper_bound_mean_list": upper_bound_mean_list[1:],
        "upper_bound_std_list": upper_bound_std_list[1:],
        "delta_log_prob_norm_list": delta_log_prob_norm_list[1:],
    }
    return result_dict

In [89]:
def plot_line(mean_list, std_list, dataset, model_name_or_path, color="#0D4C6D", label="grad", line=None, title=""):
    plt.figure(figsize=(3, 2.5))

    x = np.arange(len(mean_list))
    xticks = [str(i) for i in range(len(x))]
    step = max(1, len(x) // 4)

    mean = np.array(mean_list)
    std = np.array(std_list)
    lower = mean - std
    upper = mean + std

    plt.plot(
        x,
        mean_list,
        label=title,
        color=color,
        linewidth=2
    )
    plt.fill_between(
        x,
        lower,
        upper,
        color=color,
        alpha=0.1,
        linewidth=0
    )
    if line is not None:
        # $\|\Delta\boldsymbol{z}\|$
        plt.axhline(y=line, color="#BF1E2E", linestyle="--", linewidth=1, label=r"$\|\Delta\log \pi (y_t |\boldsymbol{h} ) \|$")

    plt.xticks(x[::step], xticks[::step])

    plt.xlabel("Number of layers")
    plt.title("")
    if line is not None:
        plt.legend()
    plt.grid(False)
    plt.tight_layout()
    
    save_path = f"../../results/figure_results/why/{model_name_or_path}/{dataset}_{label}.pdf"
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    plt.savefig(save_path)
    plt.close()

def plot_grad_and_deltaz_together(
    grad_mean_list,
    grad_std_list,
    delta_z_mean_list,
    delta_z_std_list,
    dataset,
    model_name_or_path,
    grad_color="#0D4C6D",
    delta_z_color="#FEB705",
    title=""
):
    fig, ax_left = plt.subplots(figsize=(3, 2.5))
    ax_right = ax_left.twinx()

    x_grad = np.arange(len(grad_mean_list))
    x_delta = np.arange(len(delta_z_mean_list))
    x_len = max(len(x_grad), len(x_delta))
    xticks = [str(i) for i in range(x_len)]
    step = max(1, x_len // 4)

    def draw_series(ax, x, mean_list, std_list, color, label):
        mean = np.array(mean_list)
        std = np.array(std_list)
        lower = mean - std
        upper = mean + std

        ax.plot(
            x,
            mean_list,
            label=label,
            color=color,
            linewidth=2
        )
        ax.fill_between(
            x,
            lower,
            upper,
            color=color,
            alpha=0.1,
            linewidth=0
        )

    draw_series(
        ax_left,
        x_grad,
        grad_mean_list,
        grad_std_list,
        grad_color,
        r"$\|\nabla_{\boldsymbol{h}} \log \pi (y_t|\boldsymbol{h}_0)\|$"
    )
    draw_series(
        ax_right,
        x_delta,
        delta_z_mean_list,
        delta_z_std_list,
        delta_z_color,
        r"$\|\Delta\boldsymbol{h}\|$"
    )

    ax_left.set_xticks(np.arange(x_len)[::step])
    ax_left.set_xticklabels(xticks[::step])
    ax_left.set_xlim(0, max(0, x_len - 1))

    ax_left.set_xlabel("Number of layers")
    if title:
        ax_left.set_title(title)

    ax_left.set_ylabel("")
    ax_right.set_ylabel("")

    ax_left.tick_params(axis="y", colors=grad_color)
    ax_right.tick_params(axis="y", colors=delta_z_color)
    ax_left.locator_params(axis="y", nbins=4)
    ax_right.locator_params(axis="y", nbins=6)
    ax_left.spines["left"].set_color(grad_color)
    ax_right.spines["right"].set_visible(False)
    ax_right.spines["left"].set_visible(False)
    ax_right.yaxis.set_ticks_position("left")
    ax_right.yaxis.set_label_position("left")

    handles_left, labels_left = ax_left.get_legend_handles_labels()
    handles_right, labels_right = ax_right.get_legend_handles_labels()
    if handles_left or handles_right:
        ax_left.legend(handles_left + handles_right, labels_left + labels_right)

    ax_left.grid(False)
    fig.tight_layout()

    save_path = f"../../results/figure_results/why/{model_name_or_path}/{dataset}_grad_delta_z.pdf"
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    fig.savefig(save_path)
    plt.close(fig)


In [90]:
def plot_grads_and_deltaz(dataset, model_name_or_path):
    file_path = f"../../results/data_results/real_dataset/{model_name_or_path}/{dataset}_result.jsonl"
    result_dict = get_data_list(file_path)
    
    grad_mean_list = result_dict["grad_mean_list"]
    grad_std_list = result_dict["grad_std_list"]
    delta_z_mean_list = result_dict["delta_z_mean_list"]
    delta_z_std_list = result_dict["delta_z_std_list"]
    upper_bound_mean_list = result_dict["upper_bound_mean_list"]
    upper_bound_std_list = result_dict["upper_bound_std_list"]

    delta_log_prob_norm_list = result_dict["delta_log_prob_norm_list"]
    mean_delta_log_prob_norm = np.mean(delta_log_prob_norm_list)

    # grad_color = "#0D4C6D"
    grad_color = "#00894b"
    plot_line(grad_mean_list, grad_std_list, dataset, model_name_or_path, grad_color, "grad", line=None, title=r"$\|\nabla_{\boldsymbol{h}} \log \pi (y_t|\boldsymbol{h}_0)\|$")
    delta_z_color = "#FEB705"
    plot_line(delta_z_mean_list, delta_z_std_list, dataset, model_name_or_path, delta_z_color, "delta_z", line=None, title=r"$\|\Delta\boldsymbol{h}\|$")

    plot_grad_and_deltaz_together(
        grad_mean_list,
        grad_std_list,
        delta_z_mean_list,
        delta_z_std_list,
        dataset,
        model_name_or_path,
        grad_color=grad_color,
        delta_z_color=delta_z_color
    )
    upper_bound_color = "#767676"
    plot_line(upper_bound_mean_list, upper_bound_std_list, dataset, model_name_or_path, upper_bound_color, "upper_bound", line=mean_delta_log_prob_norm, title="Upper Bound")
    


In [91]:
for dataset in dataset_list:
    for model_name_or_path in model_list:
        plot_grads_and_deltaz(dataset, model_name_or_path)
